In [1]:
import json
import pathlib
import numpy as np
import pytori as pt
import nafflib

import matplotlib.pyplot as plt
import pytori.plotting as ptplt

import xobjects as xo
import xtrack as xt

# BEAM
#=========================================================================================
nrj             = 7000e9
nemitt_x        = 2.5e-6
nemitt_y        = 2.5e-6
#-----------------------------------------------------------------------------------------
context         = xo.ContextCpu(omp_num_threads='auto')
particle_ref    = xt.Particles(p0c=nrj, q0=1, mass0=xt.PROTON_MASS_EV)
gemitt_x        = 1 if nemitt_x is None else ( nemitt_x / particle_ref.beta0[0] / particle_ref.gamma0[0])
gemitt_y        = 1 if nemitt_x is None else ( nemitt_y / particle_ref.beta0[0] / particle_ref.gamma0[0])
#=========================================================================================


# CREATING LINE
#=========================================================================================
mux         = 0.2981
twiss_init  = {'betx':180 ,'alfx': 2}
#------------------------------------------
h1  = -1j/4  # non-linear strength parameter, Henon definition
k2l = -1j*8*h1/twiss_init['betx']**(3/2)/gemitt_x**(1/2) # sextupole strength
assert np.imag(k2l) == 0, "k2l should be real"
k2l = np.real(k2l)
#=========================================================================================

length = 1
line = xt.Line( elements={
                    'start_cell': xt.Marker(),
                    '.1'        : xt.Marker(),
                    # ==========================
                    'sext'      : xt.Multipole(knl=[0,0,k2l], ksl=[0,0,0],length=0) ,
                    '.1'        : xt.Marker(),
                    'rot'       : xt.LineSegmentMap(qx=mux,qy=mux,**twiss_init,length=length),
                    # ==========================                    
                    '.2'        : xt.Marker(),
                    'end_cell'  : xt.Marker(),
                    })
#------------------------------------------
line.particle_ref   = particle_ref
line.build_tracker(_context=context)
line.freeze_longitudinal(True)
twiss   = line.twiss4d()
#=========================================================================================

Compiling ContextCpu kernels...


VerificationError: CompileError: command 'arm64-apple-darwin20.0.0-clang' failed: No such file or directory